In [0]:
%pip install databricks-feature-engineering feature-engine


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

mlflow.set_registry_uri("databricks-uc")

model_uri = "models:/feature_store.upsell.churn/3"
model_pyfunc = mlflow.pyfunc.load_model(model_uri)
run_id = model_pyfunc.metadata.run_id

model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

In [0]:
lookups = [
    FeatureLookup(table_name="feature_store.upsell.fs_geral", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_pontos", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_transacoes", lookup_key=['IdCliente', 'dtRef']),
    FeatureLookup(table_name="feature_store.upsell.fs_dia_horario", lookup_key=['IdCliente', 'dtRef']),
]

query = """
    SELECT dtRef,
            IdCliente
    FROM feature_store.upsell.fs_geral
    WHERE dtRef = (SELECT MAX(dtRef) FROM feature_store.upsell.fs_geral)
"""

df = spark.sql(query)

fe = FeatureEngineeringClient()

predict_set = fe.create_training_set(df=df, 
                                     feature_lookups=lookups, 
                                     label=None)

df_predict = predict_set.load_df().toPandas()

In [0]:
proba_churn = model.predict_proba(df_predict[model.feature_names_in_])[:,1]
df_predict["proba_churn"] = proba_churn
df_predict[['dtRef', 'IdCliente', "proba_churn"]]